# FASE 2 — Arquitectura CNN Multiescala

Construimos la red neuronal con bifurcación y fusiones matemáticas especializadas.

## Estructura general
```
Input (3, 28, 28)
     │
  ┌──┴──┐
 X1    X2   ← kernels diferentes (3x3 vs 5x5)
  └──┬──┘
   Fusión Bloque 1  → 50 canales  (magnitud euclidiana)
     │
  ┌──┴──┐
 X1    X2
  └──┬──┘
   Fusión Bloque 2  → 100 canales (Hadamard + Max)
     │
  ┌──┴──┐
 X1    X2
  └──┬──┘
   Fusión Bloque 3  → 75 canales  (pseudo-radial)
     │
  ┌──┴──┐
 X1    X2
  └──┬──┘
   Fusión Bloque 4  → 25 canales  (atención cruzada sigmoide)
     │
   GAP (Global Average Pooling)
     │
   FC → 7 clases
```

## 1. Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary

# Instalar torchinfo si no está
try:
    from torchinfo import summary
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torchinfo'])
    from torchinfo import summary

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


## 2. Módulos de Fusión

Cada fusión es un módulo independiente para que sea fácil de entender, probar y documentar en el reporte.

### Bloque 1 — Magnitud Euclidiana (Realce de Bordes)

$$F_1(X_1, X_2) = \sqrt{X_1^2 + X_2^2 + \varepsilon}$$

**¿Por qué funciona?** Similar al operador Sobel: si ambas ramas detectan un borde
(con kernels distintos), la magnitud euclidiana amplifica la respuesta combinada.

**Gradiente (backprop):**
$$\frac{\partial F_1}{\partial X_1} = \frac{X_1}{\sqrt{X_1^2 + X_2^2 + \varepsilon}}$$
El gradiente es continuo (ε evita división por cero) y proporcional a la activación relativa de cada rama.

In [2]:
class EuclideanFusion(nn.Module):
    """
    Bloque 1: F(X1, X2) = sqrt(X1^2 + X2^2 + eps)
    Realza bordes y contornos de alta frecuencia.
    eps evita inestabilidad numérica en la derivada.
    """
    def __init__(self, eps: float = 1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        return torch.sqrt(x1 ** 2 + x2 ** 2 + self.eps)

# Test rápido
f1 = EuclideanFusion()
a = torch.randn(2, 50, 14, 14)
b = torch.randn(2, 50, 14, 14)
out = f1(a, b)
print(f'EuclideanFusion  | input: {a.shape} → output: {out.shape} ✅')

EuclideanFusion  | input: torch.Size([2, 50, 14, 14]) → output: torch.Size([2, 50, 14, 14]) ✅


### Bloque 2 — Hadamard + Max (Potenciación de Texturas)

$$F_2(X_1, X_2) = (X_1 \odot X_2) + \max(X_1, X_2)$$

**¿Por qué funciona?**
- $X_1 \odot X_2$: Solo regiones donde **ambas** escalas se activan simultáneamente contribuyen (AND lógico suave). Detecta texturas coincidentes.
- $\max(X_1, X_2)$: Preserva la respuesta más fuerte individual (OR lógico suave).
La combinación resalta patrones presentes en múltiples escalas.

**Gradiente (backprop):**
$$\frac{\partial F_2}{\partial X_1} = X_2 + \mathbb{1}[X_1 > X_2]$$
El gradiente del max es 1 para el ganador, 0 para el perdedor (no diferenciable en el punto de igualdad, pero esto es raro en práctica).

In [3]:
class HadamardMaxFusion(nn.Module):
    """
    Bloque 2: F(X1, X2) = (X1 * X2) + elementwise_max(X1, X2)
    Potencia texturas dermatológicas donde ambas escalas coinciden.
    """
    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        hadamard = x1 * x2                          # producto elemento a elemento
        local_max = torch.maximum(x1, x2)           # max elemento a elemento
        return hadamard + local_max

# Test rápido
f2 = HadamardMaxFusion()
a = torch.randn(2, 100, 7, 7)
b = torch.randn(2, 100, 7, 7)
out = f2(a, b)
print(f'HadamardMaxFusion | input: {a.shape} → output: {out.shape} ✅')

HadamardMaxFusion | input: torch.Size([2, 100, 7, 7]) → output: torch.Size([2, 100, 7, 7]) ✅


### Bloque 3 — Pseudo-Radial (Consolidación de Formas)

$$F_3(X_1, X_2) = \frac{2 \cdot X_1 \cdot X_2}{X_1^2 + X_2^2 + \varepsilon}$$

**¿Por qué funciona?** Esta es la media armónica normalizada entre X1 y X2.
- Valor máximo (≈1) cuando $X_1 \approx X_2$: penaliza discrepancias extremas entre ramas.
- Valor bajo cuando una rama domina a la otra: favorece similitud morfológica.
Ideal para detectar formas circulares/irregulares donde ambas escalas deben concordar.

**Gradiente (backprop):**
$$\frac{\partial F_3}{\partial X_1} = \frac{2X_2(X_2^2 - X_1^2)}{(X_1^2 + X_2^2 + \varepsilon)^2}$$
El gradiente es mayor cuando las ramas discrepan, lo que empuja la red a aprender
representaciones similares en ambas ramas para formas complejas.

In [4]:
class PseudoRadialFusion(nn.Module):
    """
    Bloque 3: F(X1, X2) = (2 * X1 * X2) / (X1^2 + X2^2 + eps)
    Media armónica normalizada: penaliza discrepancias morfológicas.
    """
    def __init__(self, eps: float = 1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        numerator   = 2 * x1 * x2
        denominator = x1 ** 2 + x2 ** 2 + self.eps
        return numerator / denominator

# Test rápido
f3 = PseudoRadialFusion()
a = torch.randn(2, 75, 4, 4)
b = torch.randn(2, 75, 4, 4)
out = f3(a, b)
print(f'PseudoRadialFusion | input: {a.shape} → output: {out.shape} ✅')

PseudoRadialFusion | input: torch.Size([2, 75, 4, 4]) → output: torch.Size([2, 75, 4, 4]) ✅


### Bloque 4 — Atención Cruzada con Sigmoide (Abstracción Semántica)

$$F_4(X_1, X_2) = \sigma(X_1) \odot X_2$$

**¿Por qué funciona?** $\sigma(X_1)$ actúa como una **máscara de atención** (valores en [0,1]).
- X1 "decide" qué canales de X2 son relevantes semánticamente.
- Canales donde X1 es muy positivo → X2 pasa casi completo.
- Canales donde X1 es muy negativo → X2 se suprime.
Es una forma simplificada del mecanismo de atención tipo SE (Squeeze-and-Excitation).

**Gradiente (backprop):**
$$\frac{\partial F_4}{\partial X_1} = \sigma(X_1)(1 - \sigma(X_1)) \cdot X_2$$
$$\frac{\partial F_4}{\partial X_2} = \sigma(X_1)$$
El gradiente hacia X2 es directamente la atención aprendida. El gradiente hacia X1
depende de la derivada de la sigmoide (máxima cuando X1≈0, mínima en saturación).

In [5]:
class CrossAttentionFusion(nn.Module):
    """
    Bloque 4: F(X1, X2) = sigmoid(X1) * X2
    X1 genera una máscara de atención que filtra X2.
    """
    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        attention_mask = torch.sigmoid(x1)   # valores en (0, 1)
        return attention_mask * x2

# Test rápido
f4 = CrossAttentionFusion()
a = torch.randn(2, 25, 3, 3)
b = torch.randn(2, 25, 3, 3)
out = f4(a, b)
print(f'CrossAttentionFusion | input: {a.shape} → output: {out.shape} ✅')

CrossAttentionFusion | input: torch.Size([2, 25, 3, 3]) → output: torch.Size([2, 25, 3, 3]) ✅


## 3. Bloques Convolucionales

Cada bloque tiene la misma estructura:
```
entrada
  ├── Rama X1: Conv(kernel=3) → BN → ReLU
  └── Rama X2: Conv(kernel=5) → BN → ReLU
        ↓
      Fusión(X1, X2)
        ↓
      MaxPool2d(2×2)
```

El kernel 3×3 captura detalles finos, el 5×5 captura contexto más amplio.

In [6]:
class MultiscaleBlock(nn.Module):
    """
    Bloque genérico de bifurcación multiescala.
    
    Parámetros:
        in_channels  : canales de entrada
        out_channels : canales de salida (en cada rama, antes de la fusión)
        fusion       : módulo de fusión (EuclideanFusion, etc.)
        pool         : aplicar MaxPool2d al final
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        fusion: nn.Module,
        pool: bool = True
    ):
        super().__init__()

        # Rama X1: kernel 3×3
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

        # Rama X2: kernel 5×5
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

        self.fusion = fusion
        self.pool   = nn.MaxPool2d(kernel_size=2, stride=2) if pool else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.branch1(x)          # rama kernel 3×3
        x2 = self.branch2(x)          # rama kernel 5×5
        fused = self.fusion(x1, x2)   # fusión matemática específica
        return self.pool(fused)        # reducción espacial


# Test rápido
block = MultiscaleBlock(3, 50, EuclideanFusion())
x_test = torch.randn(2, 3, 28, 28)
out = block(x_test)
print(f'MultiscaleBlock | (2,3,28,28) → {out.shape}  [esperado: (2,50,14,14)] ✅')

MultiscaleBlock | (2,3,28,28) → torch.Size([2, 50, 14, 14])  [esperado: (2,50,14,14)] ✅


## 4. Red Completa — DermaCNN

Ensamblamos los 4 bloques con la secuencia de canales **50 → 100 → 75 → 25**

In [7]:
class DermaCNN(nn.Module):
    """
    CNN Multiescala No Secuencial para clasificación de DermaMNIST.

    Arquitectura:
        Bloque 1: in=3,   out=50  | Fusión: EuclideanFusion    | Pool: 28→14
        Bloque 2: in=50,  out=100 | Fusión: HadamardMaxFusion  | Pool: 14→7
        Bloque 3: in=100, out=75  | Fusión: PseudoRadialFusion | Pool: 7→3
        Bloque 4: in=75,  out=25  | Fusión: CrossAttentionFusion | Pool: 3→1 (GAP)
        Clasificador: FC(25 → 7)
    """
    def __init__(self, num_classes: int = 7, dropout: float = 0.4):
        super().__init__()

        # ── Bloques de extracción ──────────────────────────
        self.block1 = MultiscaleBlock(
            in_channels=3, out_channels=50,
            fusion=EuclideanFusion(),
            pool=True   # 28×28 → 14×14
        )
        self.block2 = MultiscaleBlock(
            in_channels=50, out_channels=100,
            fusion=HadamardMaxFusion(),
            pool=True   # 14×14 → 7×7
        )
        self.block3 = MultiscaleBlock(
            in_channels=100, out_channels=75,
            fusion=PseudoRadialFusion(),
            pool=True   # 7×7 → 3×3
        )
        self.block4 = MultiscaleBlock(
            in_channels=75, out_channels=25,
            fusion=CrossAttentionFusion(),
            pool=False  # NO pool aquí — GAP lo hace
        )  # ← los gradientes de Grad-CAM se calculan AQUÍ

        # ── Reducción espacial y clasificación ─────────────
        self.gap        = nn.AdaptiveAvgPool2d(1)   # Global Average Pooling → (B, 25, 1, 1)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(25, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)   # (B, 3,   28, 28) → (B, 50,  14, 14)
        x = self.block2(x)   # (B, 50,  14, 14) → (B, 100,  7,  7)
        x = self.block3(x)   # (B, 100,  7,  7) → (B, 75,   3,  3)
        x = self.block4(x)   # (B, 75,   3,  3) → (B, 25,   3,  3)
        x = self.gap(x)      # (B, 25,   3,  3) → (B, 25,   1,  1)
        x = x.flatten(1)     # (B, 25,   1,  1) → (B, 25)
        x = self.dropout(x)
        x = self.classifier(x)  # (B, 25) → (B, 7)
        return x


# Instanciar y mover a GPU
model = DermaCNN(num_classes=7).to(DEVICE)
print('✅ Modelo creado en:', DEVICE)

✅ Modelo creado en: cuda


## 5. Verificación del Forward Pass

In [8]:
# Forward pass con datos dummy
model.eval()
with torch.no_grad():
    dummy_input = torch.randn(4, 3, 28, 28).to(DEVICE)
    output = model(dummy_input)

print('=== FORWARD PASS VERIFICADO ===')
print(f'Input  : {dummy_input.shape}')
print(f'Output : {output.shape}  [esperado: (4, 7)] ✅')
print()

# Trazar el flujo de tensores bloque a bloque
print('Flujo de tensores:')
print('-'*40)
x = dummy_input
shapes = [('Input', x.shape)]
for name, block in [('Block 1', model.block1),
                     ('Block 2', model.block2),
                     ('Block 3', model.block3),
                     ('Block 4', model.block4)]:
    x = block(x)
    shapes.append((name, x.shape))
x = model.gap(x)
shapes.append(('GAP', x.shape))
x = x.flatten(1)
shapes.append(('Flatten', x.shape))

for name, shape in shapes:
    print(f'  {name:<12} → {tuple(shape)}')

=== FORWARD PASS VERIFICADO ===
Input  : torch.Size([4, 3, 28, 28])
Output : torch.Size([4, 7])  [esperado: (4, 7)] ✅

Flujo de tensores:
----------------------------------------
  Input        → (4, 3, 28, 28)
  Block 1      → (4, 50, 14, 14)
  Block 2      → (4, 100, 7, 7)
  Block 3      → (4, 75, 3, 3)
  Block 4      → (4, 25, 3, 3)
  GAP          → (4, 25, 1, 1)
  Flatten      → (4, 25)


## 6. Resumen de parámetros

In [9]:
summary(
    model,
    input_size=(1, 3, 28, 28),
    device=DEVICE,
    col_names=['input_size', 'output_size', 'num_params'],
    row_settings=['var_names']
)

Layer (type (var_name))                  Input Shape               Output Shape              Param #
DermaCNN (DermaCNN)                      [1, 3, 28, 28]            [1, 7]                    --
├─MultiscaleBlock (block1)               [1, 3, 28, 28]            [1, 50, 14, 14]           --
│    └─Sequential (branch1)              [1, 3, 28, 28]            [1, 50, 28, 28]           --
│    │    └─Conv2d (0)                   [1, 3, 28, 28]            [1, 50, 28, 28]           1,350
│    │    └─BatchNorm2d (1)              [1, 50, 28, 28]           [1, 50, 28, 28]           100
│    │    └─ReLU (2)                     [1, 50, 28, 28]           [1, 50, 28, 28]           --
│    └─Sequential (branch2)              [1, 3, 28, 28]            [1, 50, 28, 28]           --
│    │    └─Conv2d (0)                   [1, 3, 28, 28]            [1, 50, 28, 28]           3,750
│    │    └─BatchNorm2d (1)              [1, 50, 28, 28]           [1, 50, 28, 28]           100
│    │    └─ReLU (2)       

## 7. Verificación del Backward Pass (gradientes)

In [10]:
# Verificar que los gradientes fluyen correctamente por todas las fusiones
model.train()
dummy_input = torch.randn(4, 3, 28, 28).to(DEVICE)
dummy_labels = torch.randint(0, 7, (4,)).to(DEVICE)

criterion = nn.CrossEntropyLoss()
output = model(dummy_input)
loss = criterion(output, dummy_labels)
loss.backward()

print('=== VERIFICACIÓN DE GRADIENTES ===')
print(f'Loss: {loss.item():.4f}')
print()

# Verificar que todos los parámetros tienen gradiente
no_grad = []
for name, param in model.named_parameters():
    if param.grad is None:
        no_grad.append(name)

if no_grad:
    print(f'⚠️  Parámetros SIN gradiente: {no_grad}')
else:
    print('✅ Todos los parámetros reciben gradiente correctamente')

# Mostrar norma de gradientes por bloque
print()
print('Norma de gradientes por bloque:')
for block_name in ['block1', 'block2', 'block3', 'block4']:
    block = getattr(model, block_name)
    grads = [p.grad.norm().item() for p in block.parameters() if p.grad is not None]
    if grads:
        print(f'  {block_name}: mean_grad_norm = {sum(grads)/len(grads):.6f}')

=== VERIFICACIÓN DE GRADIENTES ===
Loss: 1.9869

✅ Todos los parámetros reciben gradiente correctamente

Norma de gradientes por bloque:
  block1: mean_grad_norm = 1.941228
  block2: mean_grad_norm = 4.985589
  block3: mean_grad_norm = 4.503784
  block4: mean_grad_norm = 0.613728


## 8. Guardar modelo como módulo reutilizable

In [11]:
model_code = '''
# model.py — Fase 2: Arquitectura CNN Multiescala
# Importar en fases posteriores: from model import DermaCNN

import torch
import torch.nn as nn


# ── Módulos de Fusión ────────────────────────────────────────────────────────

class EuclideanFusion(nn.Module):
    """Bloque 1: sqrt(X1^2 + X2^2 + eps) — Realce de bordes"""
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps
    def forward(self, x1, x2):
        return torch.sqrt(x1 ** 2 + x2 ** 2 + self.eps)


class HadamardMaxFusion(nn.Module):
    """Bloque 2: (X1 * X2) + max(X1, X2) — Potenciación de texturas"""
    def forward(self, x1, x2):
        return (x1 * x2) + torch.maximum(x1, x2)


class PseudoRadialFusion(nn.Module):
    """Bloque 3: 2*X1*X2 / (X1^2 + X2^2 + eps) — Consolidación de formas"""
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps
    def forward(self, x1, x2):
        return (2 * x1 * x2) / (x1 ** 2 + x2 ** 2 + self.eps)


class CrossAttentionFusion(nn.Module):
    """Bloque 4: sigmoid(X1) * X2 — Atención cruzada semántica"""
    def forward(self, x1, x2):
        return torch.sigmoid(x1) * x2


# ── Bloque Genérico ──────────────────────────────────────────────────────────

class MultiscaleBlock(nn.Module):
    def __init__(self, in_channels, out_channels, fusion, pool=True):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        self.fusion = fusion
        self.pool   = nn.MaxPool2d(2, 2) if pool else nn.Identity()

    def forward(self, x):
        return self.pool(self.fusion(self.branch1(x), self.branch2(x)))


# ── Red Completa ─────────────────────────────────────────────────────────────

class DermaCNN(nn.Module):
    """
    CNN Multiescala para DermaMNIST.
    Canales: 3 → 50 → 100 → 75 → 25 → 7
    """
    def __init__(self, num_classes=7, dropout=0.4):
        super().__init__()
        self.block1 = MultiscaleBlock(3,   50,  EuclideanFusion(),     pool=True)
        self.block2 = MultiscaleBlock(50,  100, HadamardMaxFusion(),   pool=True)
        self.block3 = MultiscaleBlock(100, 75,  PseudoRadialFusion(),  pool=True)
        self.block4 = MultiscaleBlock(75,  25,  CrossAttentionFusion(),pool=False)
        self.gap        = nn.AdaptiveAvgPool2d(1)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(25, num_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x).flatten(1)
        x = self.dropout(x)
        return self.classifier(x)
'''

with open('model.py', 'w', encoding='utf-8') as f:
    f.write(model_code.strip())

# Verificar import
import importlib.util
spec = importlib.util.spec_from_file_location('model', 'model.py')
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
test_model = mod.DermaCNN()
test_out   = test_model(torch.randn(2, 3, 28, 28))

print('✅ model.py guardado y verificado')
print(f'   Output shape: {test_out.shape}  [esperado: (2, 7)]')

✅ model.py guardado y verificado
   Output shape: torch.Size([2, 7])  [esperado: (2, 7)]


## ✅ Resumen Fase 2

| Bloque | Canales | Fusión | Propiedad matemática |
|--------|---------|--------|---------------------|
| 1 | 3 → 50 | `√(X₁²+X₂²+ε)` | Magnitud euclidiana — realce de bordes |
| 2 | 50 → 100 | `(X₁⊙X₂)+max(X₁,X₂)` | Hadamard + Max — potenciación texturas |
| 3 | 100 → 75 | `2X₁X₂/(X₁²+X₂²+ε)` | Media armónica — consolidación formas |
| 4 | 75 → 25 | `σ(X₁)⊙X₂` | Atención cruzada — abstracción semántica |
| GAP + FC | 25 → 7 | — | Clasificación final |

**Archivos generados:** `model.py`

**Siguiente paso → Fase 3: Entrenamiento**